# JOINT Initial Knowledge with Covariates

`InitKnowledgeStrategy.JOINT` regresses each student's logit initial knowledge on student level covariates, such as a pretest, jointly with the BKT fit:

$$\text{logit}(\pi_{know,i}) = b_0 + x_i b_1 + \sigma z_i, \quad z_i \sim \text{Normal}(0, 1)$$

This example covers:

1. Simulating data where initial knowledge depends on covariates.
2. Fitting JOINT with one and with two covariates.
3. Reading `b0`, `b1` and `sigma` from `summary()`.
4. Predicting seen and unseen students, and comparing with a model without covariates.

See the [JOINT Initial Knowledge](../guide/joint_strategy.rst) guide page for details.

In [1]:
import logging

import numpy as np
import pandas as pd
from cmdstanpy.utils import get_logger

# keep the notebook output readable, CmdStan progress is logged at INFO level
get_logger().setLevel(logging.WARNING)

## Simulate data

The simulator below draws initial knowledge from the regression, then runs the BKT process. Sequences are short (8 interactions), which is where initial knowledge matters most.

The model never sees a student's initial state directly, only noisy responses after it, so the regression needs many students. With a few hundred students the slopes tend to be overestimated. We use 800.

In [2]:
def simulate_joint(n_students, n_problems, b0, b1, sigma, learn, forget, guess, slip, seed):
    rng = np.random.default_rng(seed)
    covariates = rng.normal(size=(n_students, len(b1)))
    logit_pi = b0 + covariates @ np.asarray(b1) + sigma * rng.normal(size=n_students)
    pi_know = 1 / (1 + np.exp(-logit_pi))
    rows = []
    for s in range(n_students):
        known = rng.random() < pi_know[s]
        for t in range(n_problems):
            p_correct = 1 - slip if known else guess
            rows.append({"student_id": f"stu_{s}", "problem_id": f"prob_{t}",
                         "correct": int(rng.random() < p_correct), "timestamp": t, "kc_id": "kc_0"})
            known = rng.random() > forget if known else rng.random() < learn
    covariate_df = pd.DataFrame(covariates, columns=[f"x{j}" for j in range(len(b1))])
    covariate_df.insert(0, "student_id", [f"stu_{s}" for s in range(n_students)])
    return pd.DataFrame(rows), covariate_df


truth = {"b0": -0.5, "b1": [1.5, -1.0], "sigma": 0.3}
data_df, covariates_df = simulate_joint(
    n_students=800, n_problems=8, learn=0.1, forget=0.02, guess=0.2, slip=0.1, seed=31, **truth
)
covariates_df.head()

,student_id,x0,x1
0,stu_0,-0.395301,0.263915
1,stu_1,0.607128,-0.972160
2,stu_2,0.767664,0.255058
3,stu_3,0.782980,0.272418
4,stu_4,1.162008,-0.937756


The covariate frame has one row per student: the student ID column plus numeric covariate columns. Every column other than the ID is a covariate. The covariates here are already standardized, which keeps the default `N(0, 5)` prior on `b1` reasonable.

Hold out 160 students to test predictions for students the model has never seen.

In [3]:
students = covariates_df["student_id"].to_numpy()
rng = np.random.default_rng(0)
test_students = set(rng.choice(students, size=160, replace=False))
train_df = data_df[~data_df["student_id"].isin(test_students)]
test_df = data_df[data_df["student_id"].isin(test_students)]

## Fit with one and with two covariates

We fit JOINT with both covariates and with `x0` only, using the default priors, plus a model without covariates as a baseline.

In [4]:
from stanbkt.fits import MCMCFitOptions
from stanbkt.models import InitKnowledgeStrategy, StandardBKT, StandardPriors
from stanbkt.utils import VerbosityLevel

fit_opts = MCMCFitOptions(seed=1234, iter_warmup=500, iter_sampling=500, show_progress=False)


def joint_model():
    return StandardBKT(
        individual_initial_knowledge=True,
        init_knowledge_strategy=InitKnowledgeStrategy.JOINT,
        verbose=VerbosityLevel.WARN,
    )


joint_2 = joint_model()
joint_2.fit(train_df, student_covariates=covariates_df, stan_fit_options=fit_opts)

one_covariate = covariates_df[["student_id", "x0"]]
joint_1 = joint_model()
joint_1.fit(train_df, student_covariates=one_covariate, stan_fit_options=fit_opts)

baseline = StandardBKT(verbose=VerbosityLevel.WARN)
baseline.fit(train_df, stan_fit_options=fit_opts)

StandardBKT(fit_method=<FitMethod.MCMC: 'mcmc'>, verbose=<VerbosityLevel.WARN: 1>, is_fitted=True)

## Reading the regression

The JOINT rows of the summary are `pi_b0_know_param` (intercept), `pi_b1_know_param` (one row per covariate, labeled with the column name), `pi_sigma_param` (residual standard deviation) and `logit_pi_know_z` (one standardized residual per fitted student). All are on the logit scale.

In [5]:
regression_rows = ["pi_b0_know_param", "pi_b1_know_param", "pi_sigma_param"]
joint_2.summary().loc["kc_0"].loc[regression_rows][["Mean", "2.5%", "97.5%", "R_hat"]]

Mean      2.5%     97.5%    R_hat
parameter        axis      label                                       
pi_b0_know_param                 -0.719122 -1.023200 -0.412675  1.00164
pi_b1_know_param covariate x0     1.578250  1.235830  1.996600  1.00212
                           x1    -1.127300 -1.486100 -0.821264  1.00236
pi_sigma_param                    0.215037  0.006247  0.826296  1.02865

In [6]:
truth

{'b0': -0.5, 'b1': [1.5, -1.0], 'sigma': 0.3}

The estimates are close to the truth, and the sign of each slope tells you whether the covariate raises or lowers initial knowledge.

With only `x0`, the missing covariate shows up as a smaller `x0` slope. Leaving out a predictor shrinks the remaining slopes of a logistic regression, so include the covariates that matter.

In [7]:
joint_1.summary().loc["kc_0"].loc[regression_rows][["Mean", "2.5%", "97.5%"]]

,,,Mean,2.5%,97.5%
parameter,axis,label,,,
pi_b0_know_param,,,-0.611963,-0.868550,-0.36668
pi_b1_know_param,covariate,x0,1.208560,0.948918,1.50178
pi_sigma_param,,,0.194646,0.003957,0.74010


## Why the prior on sigma matters

Each student contributes a single binary piece of information about their initial knowledge (whether they started in the known state), and a logistic regression with a per student residual is only weakly identified from one binary outcome per student: a larger `sigma` with proportionally larger slopes predicts almost the same probabilities. The data cannot tell them apart, so the prior on `sigma` decides where the fit lands.

The default `pi_sigma_lambda=5` is an Exponential prior with mean 0.2, which assumes the covariates explain most of the differences between students. Here is the same fit with a loose prior, `pi_sigma_lambda=0.5` (mean 2):

In [8]:
joint_loose = joint_model()
joint_loose.fit(train_df, student_covariates=covariates_df, priors=StandardPriors(pi_sigma_lambda=0.5),
                stan_fit_options=fit_opts)
joint_loose.summary().loc["kc_0"].loc[regression_rows][["Mean", "2.5%", "97.5%", "R_hat"]]

Mean      2.5%     97.5%    R_hat
parameter        axis      label                                      
pi_b0_know_param                 -1.68285 -3.778850 -0.615119  1.03245
pi_b1_know_param covariate x0     3.67448  1.509840  7.946710  1.03372
                           x1    -2.62155 -5.768590 -1.026550  1.03730
pi_sigma_param                    3.49948  0.379939  8.534250  1.04186

The signs are still right, but the slopes and `sigma` are inflated well beyond the truth, and `R_hat` above 1.01 shows the chains wandering along the ridge between the two explanations. Loosen the prior only if you expect large differences between students that the covariates do not capture, and fit with more than one `sigma` prior when the size of the slopes matters for your conclusions.

## Predicting seen and unseen students

Pass `student_covariates` to every prediction method as well. Students in the fit reuse their fitted residual `z`. Unseen students, like the held out ones here, get the regression mean $b_0 + x_i b_1$.

Initial knowledge mostly affects the first few interactions, so we score the first three interactions of the held out students.

In [9]:
from stanbkt.utils import rmse

test_df = test_df.assign(position=test_df.groupby("student_id").cumcount())
early = test_df[test_df["position"] < 3]

scores = {}
for name, model, covs in [("no covariates", baseline, None),
                          ("JOINT, x0 and x1, loose sigma prior", joint_loose, covariates_df),
                          ("JOINT, x0", joint_1, one_covariate),
                          ("JOINT, x0 and x1", joint_2, covariates_df)]:
    kwargs = {} if covs is None else {"student_covariates": covs}
    preds = model.predict(early, **kwargs)
    scores[name] = rmse(preds["correct"], preds["pCorrectness"])
pd.Series(scores, name="RMSE, first 3 interactions of unseen students").round(4)

no covariates                          0.4560
JOINT, x0 and x1, loose sigma prior    0.4281
JOINT, x0                              0.4344
JOINT, x0 and x1                       0.4265
Name: RMSE, first 3 interactions of unseen students, dtype: float64

The covariates let the model start each new student at a sensible level instead of the population average, which lowers the error on early interactions.

Without covariates for a student, prediction raises a `ValueError`, so make sure the frame covers every student in the prediction data:

In [10]:
try:
    joint_2.predict(test_df, student_covariates=covariates_df.iloc[:10])
except ValueError as e:
    print("ValueError:", str(e)[:120], "...")

ValueError: 'student_covariates' is missing rows for student(s) present in the interaction data: ['stu_10', 'stu_113', 'stu_118', 's ...
